In [1]:
import pandas as pd
import numpy as np
import warnings
from evcouplings.compare import DistanceMap
warnings.filterwarnings("ignore")

genome_position_to_gene = pd.read_csv("../../databases/genome_position_to_gene_with_codons.csv")

## Prepare the master table of SNPs and indels

In [2]:
# Read in indel matrix from Roger
# pandas uses xlrd to read excel files, but xlrd removed support for xlsx files, 
# because of course they don't need the most useful file type...
# indels = pd.read_excel("../../databases/INDELs homoplasy count across lineages with spacer 4.xlsx", engine='openpyxl')
# indels.columns = [x.replace(" ", "_") for x in indels.columns]
# indels.to_csv("../../databases/INDELs homoplasy count across lineages with spacer 4.csv", index=None)

indels = pd.read_csv("../../databases/INDELs homoplasy count across lineages with spacer 4.csv")
print("total number of indels", len(indels))

# Restrict to genes only
indels = indels.query("Gene_Coord != 'None'")
print("Number of indels in a gene", len(indels))

indels = indels.query("INDEL_type != 'frameshift'")
print("Number of non-frameshift indels", len(indels))

indels.rename({"ins/del":"ins_del"}, axis=1, inplace=True)
deletions = indels.query("INDEL_type=='inframe' and ins_del=='del'")
insertions = indels.query("INDEL_type=='inframe' and ins_del=='ins'")

print("Number of inframe deletions", len(deletions))
print("Number of inframe insertions", len(insertions))

grouped_deletions = deletions.groupby(["H37Rv_Locus_Tag", "Codon_Pos", "Position"]).sum()
grouped_deletions = pd.DataFrame(grouped_deletions).reset_index()
grouped_deletions = grouped_deletions.rename({"total":"num_inframe_deletions",
                                                "Codon_Pos":"codon",
                                                "Position":"i",
                                                "H37Rv_Locus_Tag":"locus"
                                               }, axis=1)
grouped_deletions = grouped_deletions[["i", "codon", "locus", "num_inframe_deletions"]]

grouped_insertions = insertions.groupby(["H37Rv_Locus_Tag", "Codon_Pos","Position"]).sum()
grouped_insertions = pd.DataFrame(grouped_insertions).reset_index()
grouped_insertions = grouped_insertions.rename({"total":"num_inframe_insertions",
                                                "Codon_Pos":"codon",
                                                "Position":"i",
                                                "H37Rv_Locus_Tag":"locus"
                                               }, axis=1)
grouped_insertions = grouped_insertions[["i", "codon", "locus", "num_inframe_insertions"]]




total number of indels 46306
Number of indels in a gene 38619
Number of non-frameshift indels 5678
Number of inframe deletions 4137
Number of inframe insertions 1541


In [3]:
## Merge to create the master indel table!
inframe_insertion_count = grouped_insertions
inframe_deletion_count = grouped_deletions

master_indel_count = inframe_insertion_count.merge(inframe_deletion_count, on=['i', 'locus', 'codon'], how="outer")

master_indel_count.fillna(0, inplace=True)
mutations = np.array([master_indel_count.num_inframe_deletions, master_indel_count.num_inframe_insertions]).T

master_indel_count["total_indels"] = np.sum(mutations, axis=1)
master_indel_count = master_indel_count.query("total_indels > 0")
master_indel_count.sort_values("total_indels", ascending=False)

master_indel_count.to_csv("summary_data/master_indel_count_table.csv")

## Combine indels and mutations into one table

In [4]:
snps = pd.read_pickle("/n/data1/hms/dbmi/farhat/anna/structure_annotation/homoplasy_count_across_lineages.pkl")

# ## Filter out intergenic SNPs

snps = snps.query("gene_id != 'intergenic' and AA_change!='None' and SNP_type=='N'")
print(len(snps))
snps["residue"] = [int("".join(list(x)[1:-1])) for x in snps.AA_change]
snps["summed"] = snps["total"]
# ## Merge with indels

indels = pd.read_csv("summary_data/master_indel_count_table.csv", index_col=0)

indels = indels.query("total_indels > 0")
indels = indels.rename({
    "i": "pos", 
    "locus": "gene_id",
    "total_indels": "summed",
    "codon":"residue"
}, axis=1)
indels["SNP_type"] = "indel"

snps.to_csv("summary_data/02.SNPs_in_analysis.csv")
indels.to_csv("summary_data/02.Indels_in_analysis.csv")

snps.pos = snps.pos.astype(int)
all_mutations = pd.concat([snps, indels])
all_mutations.columns

all_mutations = all_mutations[[
    "pos","ref", "alt", "gene_id", "SNP_type", "summed", "residue", "num_inframe_insertions", "num_inframe_deletions"
]]
all_mutations.to_csv("summary_data/02.all_mutations_in_analysis_prefiltering.csv")



477607


In [5]:
print(len(all_mutations)), print(len(snps)), print(len(indels))

483117
477607
5510


(None, None, None)

#### Filter out the EBR and mappability regions
#### also the blindspots analysis

In [6]:
# Ensure that we've accurately filtered positions from Marin et al
ebr = pd.read_csv(
    "input/EBR.csv", index_col=0
)
# Accurately filter the positions from Modlin et al
modlin = pd.read_csv("input/S7-seq-attributes.csv")[["rv_position", "blindspot"]]

# Filter proteins with low overall mapabbility and EBR
mappability = pd.read_csv("input/AF7_H37Rv_FeatureLevelAnalysis.EBR_And_Pmap.tsv", sep="\t")
mappability = mappability.query('Feature == "CDS"')

proteins_to_keep = mappability.query("Mean_EBR_36CI >= .9 and Mean_PileupMapScore_K50_E4 >= .9")

# Filter out the positions
all_mutations = pd.read_csv("summary_data/02.all_mutations_in_analysis_prefiltering.csv", index_col=0)
print(len(all_mutations))
all_mutations = all_mutations.merge(ebr, how="left", left_on="pos", right_on="position")
all_mutations = all_mutations.merge(modlin, how="left", left_on="pos", right_on="rv_position")
all_mutations = all_mutations.query("EBR >=0.9 and blindspot==0 and gene_id in @proteins_to_keep.H37rv_GeneID")


483117


In [7]:
all_mutations.to_csv("summary_data//02.all_mutations_in_analysis_filtered.csv")
len(all_mutations), len(all_mutations.query("SNP_type=='indel'")), len(all_mutations.query("SNP_type=='N'"))

(475046, 5104, 469942)

## Filter for just non-synonymous changes

In [7]:
def mutation_table_cleanup(all_mutations, locus_id, nonsynonymous_only=True):
    
    # prepare the SNP input table
    mutation_count_table = all_mutations.query('gene_id==@locus_id')
    
    # eliminate non-synonymouse mutations, if chosen
    if nonsynonymous_only:
        mutation_count_table= mutation_count_table.query("SNP_type!='S'")
    
    #Sum the number of mutations per columns
    mutation_count_table['summed'] = mutation_count_table.summed.astype(float)
    mutation_count_table = mutation_count_table.groupby("residue").sum()
    mutation_count_table.index.rename("", inplace=True)
    mutation_count_table["residue"] = mutation_count_table.index
    mutation_count_table['residue'] = mutation_count_table.residue.astype(int)
    
    #information about individual positions is irrelevant after summation

    mutation_count_table = mutation_count_table[[
        'residue', 'summed'
    ]]
    
    return mutation_count_table


In [8]:
all_hits = pd.read_csv("summary_data/01.structure_hits.csv", index_col=0)
proteins_excluded = []

analyzed_mutations = []

# Determine if there are enough NS mutations to perform this analysis
for idx, row in all_hits.iterrows():
    
    uid = row.uid
    
    to_analyzer = mutation_table_cleanup(all_mutations, row.rv, nonsynonymous_only=True)
    
    if len(to_analyzer) < 2:
        print("too few mutations", uid)
        proteins_excluded.append(uid)
        continue

    # read in the distance map to ensure that all residues analyzed have a structure known
    dm = DistanceMap.from_file(f"filtered_distmaps/{row.Entry}")
    dm.residues_i['id'] = dm.residues_i.id.astype(int)
    to_analyze = to_analyzer.query('residue in @dm.residues_i.id')

    to_analyze.to_csv(f"output/{row.uid}/all_mutations_analyzed.csv")
    to_analyze["uid"] = row.uid
    to_analyze["rv"] = row.rv
    to_analyze["Entry"] = row.Entry

    analyzed_mutations.append(to_analyze)


too few mutations Y2645_MYCTU
too few mutations Y3428_MYCTU
too few mutations Y954A_MYCTU
too few mutations VPC16_MYCTU
too few mutations MFTA_MYCTU
too few mutations I6X8E6_MYCTU
too few mutations I6Y560_MYCTU
too few mutations O86339_MYCTU
too few mutations O50414_MYCTU
too few mutations P95131_MYCTU
too few mutations Q10844_MYCTU
too few mutations I6Y2Q7_MYCTU
too few mutations I6YAC9_MYCTU
too few mutations I6X9N4_MYCTU
too few mutations P71601_MYCTU
too few mutations O06797_MYCTU
too few mutations I6X043_MYCTU
too few mutations I6XEK2_MYCTU
too few mutations I6Y1F0_MYCTU
too few mutations I6X630_MYCTU
too few mutations I6Y8X8_MYCTU
too few mutations O06370_MYCTU
too few mutations P71949_MYCTU
too few mutations P96366_MYCTU
too few mutations Q79FL0_MYCTU
too few mutations O53402_MYCTU
too few mutations O50413_MYCTU
too few mutations O53337_MYCTU
too few mutations Q10843_MYCTU
too few mutations I6YC39_MYCTU
too few mutations P71641_MYCTU
too few mutations P71639_MYCTU
too few mutati

In [15]:
print(len(proteins_excluded), "proteins have too few mutations")

89 proteins have too few mutations


In [16]:
all_hits = all_hits.query("not uid in @proteins_excluded")
all_hits.to_csv("summary_data/02.structure_hits_filtered.csv")

In [17]:
all_mutations

,pos,ref,alt,gene_id,SNP_type,summed,residue,num_inframe_insertions,num_inframe_deletions
0,2155168,C,G,Rv1908c,N,1717.0,315,NaN,NaN
1,761155,C,T,Rv0667,N,1406.0,450,NaN,NaN
2,781687,A,G,Rv0682,N,779.0,43,NaN,NaN
4,4247429,A,G,Rv3795,N,743.0,306,NaN,NaN
6,7582,A,G,Rv0006,N,552.0,94,NaN,NaN
...,...,...,...,...,...,...,...,...,...
5505,4401556,NaN,NaN,Rv3912,indel,1.0,229,0.0,1.0
5506,4406378,NaN,NaN,Rv3917c,indel,1.0,38,0.0,1.0
5507,4408890,NaN,NaN,Rv3920c,indel,2.0,3,0.0,2.0
5508,4409222,NaN,NaN,Rv3921c,indel,9.0,283,0.0,9.0


In [12]:
analyzed_mutations_df = pd.concat(analyzed_mutations)
analyzed_mutations_df.to_csv("summary_data/02.all_mutations_analyzed.csv")

In [13]:
len(analyzed_mutations_df)

335419

In [14]:
analyzed_mutations_df

,residue,summed,uid,rv,Entry
,,,,,
124,124,1.0,Y876_MYCTU,Rv0876c,P9WKR5
125,125,1.0,Y876_MYCTU,Rv0876c,P9WKR5
126,126,1.0,Y876_MYCTU,Rv0876c,P9WKR5
132,132,5.0,Y876_MYCTU,Rv0876c,P9WKR5
138,138,1.0,Y876_MYCTU,Rv0876c,P9WKR5
...,...,...,...,...,...
329,329,1.0,O53340_MYCTU,Rv3194c,O53340
330,330,2.0,O53340_MYCTU,Rv3194c,O53340
333,333,1.0,O53340_MYCTU,Rv3194c,O53340
